# 03 — Train Music Transformer

Run `00_setup_and_data.ipynb` first. Needs `TRAIN_CSV` and `VAL_CSV`.


In [ ]:
import os
MIDI_DIR   = '/content/maestro'
TRAIN_CSV  = os.path.join(MIDI_DIR, 'train.csv')
VAL_CSV    = os.path.join(MIDI_DIR, 'val.csv')
OUTPUT_DIR = '/content/drive/MyDrive/deep-techno-data/checkpoints/transformer'

## Config


In [ ]:
from deepTechno.training.train_transformer import TransformerConfig

config = TransformerConfig(
    train_csv      = TRAIN_CSV,
    val_csv        = VAL_CSV,
    output_dir     = OUTPUT_DIR,
    n_layers       = 6,
    num_heads      = 8,
    d_model        = 512,
    dim_feedforward= 1024,
    dropout        = 0.1,
    max_sequence   = 2048,
    rpr            = True,
    batch_size     = 2,
    epochs         = 100,
    ce_smoothing   = 0.1,
)
print(config)

## Train


In [ ]:
from deepTechno.training.train_transformer import run_transformer_training
model = run_transformer_training(config)

## Generate from validation primer


In [ ]:
from deepTechno.generation.generate_transformer import generate_from_dataset

tokens = generate_from_dataset(
    model, VAL_CSV,
    num_primer=256,
    target_seq_length=1024,
    beam=0,
    out_dir='/content/generated_transformer',
)
print(f'Generated {len(tokens)} tokens')

## Load a saved checkpoint


In [ ]:
# To resume from a saved weights file:
# from deepTechno.generation.generate_transformer import load_model
# model = load_model(f'{OUTPUT_DIR}/best_model.pt', rpr=True)
print('Uncomment the block above after saving a checkpoint')

In [ ]:
from IPython.display import Audio
import glob
mid_files = sorted(glob.glob('/content/generated_transformer/*.mid'))
if mid_files:
    try:
        import subprocess
        subprocess.run(['apt-get', 'install', '-qq', 'fluidsynth'], check=True)
        wav = '/content/transformer_out.wav'
        subprocess.run([
            'fluidsynth', '-ni', '/usr/share/sounds/sf2/default-GM.sf2',
            mid_files[-1], '-F', wav, '-r', '44100'
        ], capture_output=True)
        display(Audio(wav))
    except Exception as e:
        print('Audio unavailable:', e)
else:
    print('No MIDI generated yet.')